In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="1sQVwGaYus1fScW0sxmnCp1dsOrNdCofS", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/05_01_intro.mp3"))

In [ ]:
# 🔧 Setup: Run this cell first!
# Check GPU availability and install dependencies

import torch
import sys

# Check GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("⚠️ No GPU detected. Some cells may run slowly.")
    print("   Go to Runtime → Change runtime type → GPU")

print(f"\n📦 Python {sys.version.split()[0]}")
print(f"🔥 PyTorch {torch.__version__}")

# Set random seeds for reproducibility
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"🎲 Random seed set to {SEED}")

%matplotlib inline

# Session Management & Forking — Hands-On Lab

Welcome to your fifth hands-on lab for Domain 1 of the Claude Certified Architect exam. In this notebook, we will **build a complete session management system from scratch** — implementing resume, fork, crash recovery, and the critical judgment call between resuming a stale session and starting fresh with an injected summary.

By the end of this notebook, you will be able to:
- Implement `--resume` with named sessions that restore full conversation history
- Build `fork_session` to create parallel exploration branches
- Handle crash recovery by detecting and restoring interrupted sessions
- Decide when to resume vs start fresh with an injected summary

**Estimated time: 50–60 minutes**

## AI Teaching Assistant

Need help with this notebook? Open the **AI Teaching Assistant** — it has already read this entire notebook and can help with concepts, code, and exercises.

**[Open AI Teaching Assistant](https://pods.vizuara.ai/courses/claude-certified-architect/agentic-architecture/practice/5/assistant)**

*Tip: Open it in a separate tab and work through this notebook side-by-side.*

In [ ]:
#@title 🎧 Listen: Why Matters
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_02_why_matters.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 1. Why Does This Matter?

Imagine you have been debugging a complex authentication issue for 45 minutes. Your agent has read 12 files, identified 3 suspicious patterns, and is about to test a fix. Then your laptop crashes. Without session management, you start from zero — 45 minutes of context and reasoning vanished.

Or consider a different scenario: you are exploring two possible fixes for a bug. Approach A involves refactoring the parser. Approach B involves adding input validation. You want to try both in parallel without one contaminating the other. Without session forking, you would have to try them sequentially, losing the ability to compare cleanly.

Session management is the infrastructure that makes long-running and exploratory agent work reliable. The exam tests three specific capabilities: resuming named sessions, forking sessions for parallel exploration, and knowing when stale context makes resuming dangerous. Let us build all three.

In [ ]:
#@title 🎧 Listen: Building Intuition
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_03_building_intuition.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 2. Building Intuition — Save Points in a Video Game

Think of session management like save points in a video game:

- **Resume** = loading a save file and continuing from where you stopped
- **Fork** = saving your game, then playing two different strategies from the same save point
- **Stale context** = loading a save file from 6 months ago when the game has been patched — your inventory might reference items that no longer exist

In [ ]:
# Setup
import json
import uuid
import copy
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Tuple
from datetime import datetime, timezone, timedelta

# Video game analogy
print("SESSION MANAGEMENT ANALOGY:")
print()
print("  Resume (--resume 'debug-auth'):")
print("    Like loading a save file — exact same state, pick up where you left off")
print()
print("  Fork (fork_session):")
print("    Like saving your game, then trying two different boss strategies")
print("    Save point → Branch A: aggressive strategy")
print("                 Branch B: defensive strategy")
print("    Compare results, pick the winner")
print()
print("  Stale Context:")
print("    Like loading a save from before a game patch")
print("    Your items might reference things that no longer exist")
print("    Solution: start a new game with a 'recap' of your progress")

In [ ]:
#@title 🎧 Listen: Core Implementation
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_04_core_implementation.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 3. Core Implementation — Session Manager

In [ ]:
@dataclass
class Message:
    """A single message in the conversation history."""
    role: str  # "user", "assistant", or "system"
    content: Any
    timestamp: str = field(default_factory=lambda: datetime.now(timezone.utc).isoformat())

@dataclass
class ToolResult:
    """A cached tool result from a session."""
    tool_name: str
    tool_input: Dict
    result: str
    timestamp: str = field(default_factory=lambda: datetime.now(timezone.utc).isoformat())
    source_file: Optional[str] = None

@dataclass
class Session:
    """A complete agent session with conversation history and metadata."""
    session_id: str = field(default_factory=lambda: uuid.uuid4().hex[:8])
    name: str = ""
    messages: List[Message] = field(default_factory=list)
    tool_results: List[ToolResult] = field(default_factory=list)
    files_read: List[str] = field(default_factory=list)
    status: str = "active"  # active, completed, crashed, forked
    created_at: str = field(default_factory=lambda: datetime.now(timezone.utc).isoformat())
    last_active: str = field(default_factory=lambda: datetime.now(timezone.utc).isoformat())
    parent_session_id: Optional[str] = None  # set when forked
    fork_point_index: Optional[int] = None   # message index where fork occurred
    metadata: Dict[str, Any] = field(default_factory=dict)

    def add_message(self, role: str, content: Any):
        self.messages.append(Message(role=role, content=content))
        self.last_active = datetime.now(timezone.utc).isoformat()

    def add_tool_result(self, tool_name: str, tool_input: Dict, result: str,
                        source_file: Optional[str] = None):
        self.tool_results.append(ToolResult(
            tool_name=tool_name, tool_input=tool_input,
            result=result, source_file=source_file
        ))

    def get_summary(self) -> str:
        """Generate a summary of this session for injection into a new session."""
        msg_count = len(self.messages)
        tool_count = len(self.tool_results)
        files = self.files_read

        summary_parts = [
            f"Previous session '{self.name}' (ID: {self.session_id}):",
            f"- {msg_count} messages exchanged",
            f"- {tool_count} tool calls made",
            f"- Files read: {', '.join(files) if files else 'none'}",
        ]

        # Add key findings from assistant messages
        findings = []
        for msg in self.messages:
            if msg.role == "assistant" and isinstance(msg.content, str):
                if len(msg.content) > 50:
                    findings.append(msg.content[:100] + "...")
        if findings:
            summary_parts.append("- Key findings:")
            for f in findings[-3:]:  # Last 3 findings
                summary_parts.append(f"  * {f}")

        return "\n".join(summary_parts)


class SessionManager:
    """Manages agent sessions — create, resume, fork, and recover."""

    def __init__(self):
        self.sessions: Dict[str, Session] = {}
        self.name_index: Dict[str, str] = {}  # name → session_id

    def create_session(self, name: str = "", verbose: bool = True) -> Session:
        """Create a new named session."""
        session = Session(name=name or f"session-{uuid.uuid4().hex[:4]}")
        self.sessions[session.session_id] = session
        self.name_index[session.name] = session.session_id

        if verbose:
            print(f"Created session '{session.name}' (ID: {session.session_id})")
        return session

    def resume_session(self, name: str, verbose: bool = True) -> Optional[Session]:
        """Resume a named session — equivalent to --resume flag."""
        session_id = self.name_index.get(name)
        if not session_id:
            if verbose:
                print(f"Session '{name}' not found.")
            return None

        session = self.sessions[session_id]
        session.status = "active"
        session.last_active = datetime.now(timezone.utc).isoformat()

        if verbose:
            print(f"Resumed session '{name}' (ID: {session_id})")
            print(f"  Messages: {len(session.messages)}")
            print(f"  Tool results: {len(session.tool_results)}")
            print(f"  Files read: {session.files_read}")
            print(f"  Created: {session.created_at}")

        return session

    def fork_session(self, source_name: str, fork_name: str = "",
                     verbose: bool = True) -> Optional[Session]:
        """Fork a session — create a branch with copied context."""
        source_id = self.name_index.get(source_name)
        if not source_id:
            if verbose:
                print(f"Source session '{source_name}' not found.")
            return None

        source = self.sessions[source_id]

        # Deep copy the session state
        forked = Session(
            name=fork_name or f"{source.name}-fork-{uuid.uuid4().hex[:4]}",
            messages=copy.deepcopy(source.messages),
            tool_results=copy.deepcopy(source.tool_results),
            files_read=source.files_read.copy(),
            status="active",
            parent_session_id=source.session_id,
            fork_point_index=len(source.messages),
            metadata={"forked_from": source.name},
        )

        # Mark source as forked
        source.metadata.setdefault("forks", []).append(forked.session_id)

        self.sessions[forked.session_id] = forked
        self.name_index[forked.name] = forked.session_id

        if verbose:
            print(f"Forked session '{source.name}' → '{forked.name}'")
            print(f"  Copied: {len(forked.messages)} messages, "
                  f"{len(forked.tool_results)} tool results")
            print(f"  Fork point: message index {forked.fork_point_index}")

        return forked

    def detect_stale_context(self, session: Session,
                              changed_files: List[str]) -> Dict:
        """Check if a session's context is stale due to file changes."""
        stale_files = [f for f in session.files_read if f in changed_files]
        stale_tool_results = [
            tr for tr in session.tool_results
            if tr.source_file in changed_files
        ]

        is_stale = len(stale_files) > 0 or len(stale_tool_results) > 0

        return {
            "is_stale": is_stale,
            "stale_files": stale_files,
            "stale_tool_results": len(stale_tool_results),
            "recommendation": "start_fresh_with_summary" if is_stale else "safe_to_resume",
            "message": (
                f"WARNING: {len(stale_files)} files changed since last session. "
                f"{len(stale_tool_results)} tool results may be outdated. "
                f"Recommend starting fresh with an injected summary."
                if is_stale else
                "Session context is current. Safe to resume."
            ),
        }

    def recover_crashed_session(self, name: str,
                                 verbose: bool = True) -> Optional[Session]:
        """Recover a session that was interrupted (crashed)."""
        session_id = self.name_index.get(name)
        if not session_id:
            if verbose:
                print(f"Session '{name}' not found for recovery.")
            return None

        session = self.sessions[session_id]
        if session.status != "crashed":
            if verbose:
                print(f"Session '{name}' is not crashed (status: {session.status})")

        session.status = "active"

        # Check for incomplete tool calls (assistant message with no following tool_result)
        incomplete = []
        for i, msg in enumerate(session.messages):
            if msg.role == "assistant" and isinstance(msg.content, list):
                for block in msg.content:
                    if isinstance(block, dict) and block.get("type") == "tool_use":
                        # Check if next message is the corresponding tool_result
                        has_result = (
                            i + 1 < len(session.messages) and
                            session.messages[i + 1].role == "user"
                        )
                        if not has_result:
                            incomplete.append(block.get("name", "unknown"))

        if verbose:
            print(f"Recovered session '{name}'")
            print(f"  Messages: {len(session.messages)}")
            if incomplete:
                print(f"  Incomplete tool calls found: {incomplete}")
                print(f"  These tools were called but never returned results.")
                print(f"  The agent should re-execute these tools.")
            else:
                print(f"  No incomplete tool calls — clean recovery.")

        session.metadata["recovered"] = True
        session.metadata["incomplete_tools"] = incomplete
        return session

    def list_sessions(self):
        """List all sessions with their status."""
        print(f"\n{'Name':<25} {'Status':<12} {'Messages':<10} {'Tools':<8} {'Parent'}")
        print("=" * 75)
        for sid, session in self.sessions.items():
            parent = session.metadata.get("forked_from", "-")
            print(f"  {session.name:<23} {session.status:<12} "
                  f"{len(session.messages):<10} {len(session.tool_results):<8} {parent}")

manager = SessionManager()
print("SessionManager created!")

In [ ]:
#@title 🎧 Listen: Lifecycle Demo
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_05_lifecycle_demo.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Demonstrate the full session lifecycle

# 1. Create a session
session = manager.create_session("debug-auth-module")

# 2. Simulate agent work
session.add_message("user", "Find and fix the authentication bug causing intermittent failures")
session.add_message("assistant", "I will start by reading the auth module files to understand the codebase structure.")
session.files_read.append("src/auth/login.py")
session.add_tool_result("read_file", {"path": "src/auth/login.py"},
                        '{"content": "def login(user, password): ..."}',
                        source_file="src/auth/login.py")
session.add_message("assistant", "Found suspicious retry logic in login.py — no exponential backoff. Checking token.py next.")
session.files_read.append("src/auth/token.py")
session.add_tool_result("read_file", {"path": "src/auth/token.py"},
                        '{"content": "def refresh_token(): cached = get_cache()..."}',
                        source_file="src/auth/token.py")
session.add_message("assistant", "Root cause identified: token.py serves stale tokens from cache without checking expiry.")

print(f"\nSession state after agent work:")
print(f"  Messages: {len(session.messages)}")
print(f"  Tool results: {len(session.tool_results)}")
print(f"  Files read: {session.files_read}")

# 3. Simulate a break — user closes the terminal
session.status = "completed"
print(f"\nSession paused (status: {session.status})")

# 4. Resume later
print(f"\n--- Later that day ---\n")
resumed = manager.resume_session("debug-auth-module")
if resumed:
    print(f"\nResumed! Last message was:")
    print(f"  '{resumed.messages[-1].content}'")

In [ ]:
#@title 🎧 Listen: Stale Context
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_06_stale_context.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 4. Informing Resumed Sessions About Changes

When files change between sessions, the agent's cached tool results become stale. The exam specifically tests whether you know to inform the agent about these changes.

In [ ]:
# Simulate: files changed while the session was inactive

changed_files = ["src/auth/login.py", "src/auth/session.py"]

print("Checking for stale context...")
stale_check = manager.detect_stale_context(session, changed_files)
print(f"\n  Is stale: {stale_check['is_stale']}")
print(f"  Stale files: {stale_check['stale_files']}")
print(f"  Stale tool results: {stale_check['stale_tool_results']}")
print(f"  Recommendation: {stale_check['recommendation']}")
print(f"\n  {stale_check['message']}")

# The correct way to inform a resumed session about changes
if stale_check["is_stale"]:
    change_notification = (
        "Since your last session, these files were modified:\n"
        + "\n".join(f"- {f}" for f in changed_files)
        + "\nPlease re-read these files before continuing."
    )
    print(f"\nChange notification to inject:")
    print(f"  {change_notification}")

    # In a real system, you would prepend this to the first message
    session.add_message("user", change_notification)
    print(f"\nNotification added as user message. Agent will re-read files.")

In [ ]:
#@title 🎧 Listen: Forking
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_07_forking.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 5. Fork Session for Parallel Exploration

Now let us implement the fork pattern. We will explore two different fix strategies for the same bug.

In [ ]:
# Fork the session to try two different approaches
print("Creating two parallel exploration branches...\n")

# Branch A: Refactoring approach
branch_a = manager.fork_session("debug-auth-module", "fix-via-refactor")
if branch_a:
    branch_a.add_message("user", "Fix the auth bug by refactoring the token refresh logic. "
                         "Replace the cache-first approach with a check-then-cache pattern.")
    branch_a.add_message("assistant", "I will refactor token.py to validate token expiry before returning cached values.")
    branch_a.add_tool_result("edit_file", {"path": "src/auth/token.py", "changes": "added expiry check"},
                            '{"success": true, "lines_changed": 15}',
                            source_file="src/auth/token.py")
    branch_a.add_message("assistant", "Refactored token.py: added expiry validation before cache lookup. "
                         "15 lines changed. Running tests...")
    branch_a.add_tool_result("run_tests", {"suite": "auth"},
                            '{"passed": 42, "failed": 0}',
                            source_file=None)
    branch_a.add_message("assistant", "All 42 auth tests pass. Fix complete via refactoring.")

print()

# Branch B: Validation approach
branch_b = manager.fork_session("debug-auth-module", "fix-via-validation")
if branch_b:
    branch_b.add_message("user", "Fix the auth bug by adding input validation. "
                         "Add a middleware layer that validates tokens before they reach the auth module.")
    branch_b.add_message("assistant", "I will create a new validation middleware that checks token freshness.")
    branch_b.add_tool_result("create_file", {"path": "src/auth/middleware_validation.py"},
                            '{"success": true, "lines_written": 35}',
                            source_file="src/auth/middleware_validation.py")
    branch_b.add_message("assistant", "Created middleware_validation.py (35 lines). "
                         "Running tests...")
    branch_b.add_tool_result("run_tests", {"suite": "auth"},
                            '{"passed": 40, "failed": 2}',
                            source_file=None)
    branch_b.add_message("assistant", "40 of 42 tests pass. 2 edge case failures in token expiry handling. "
                         "The middleware approach works but needs additional edge case coverage.")

# Compare branches
print("\n" + "="*60)
print("BRANCH COMPARISON")
print("="*60)
print()

for branch, label in [(branch_a, "Refactor"), (branch_b, "Validation")]:
    if branch:
        # Find test results
        test_result = None
        for tr in branch.tool_results:
            if tr.tool_name == "run_tests":
                test_result = json.loads(tr.result)
        last_msg = branch.messages[-1].content if branch.messages else "N/A"
        print(f"  {label} ({branch.name}):")
        print(f"    Messages: {len(branch.messages)}")
        print(f"    Tool calls: {len(branch.tool_results)}")
        if test_result:
            print(f"    Tests: {test_result['passed']} passed, {test_result['failed']} failed")
        print(f"    Result: {last_msg[:80]}...")
        print()

# The original session is unaffected
print("Original session (debug-auth-module):")
print(f"  Messages: {len(session.messages)} (unchanged by forks)")
print(f"  Status: {session.status}")

Notice how each fork evolved independently. Branch A refactored existing code and all tests pass. Branch B added new code but has 2 test failures. The original session was never modified — each branch is a completely isolated copy of the context at the fork point.

In [ ]:
#@title 🎧 Listen: Exercise1
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_08_exercise1.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 6. Exercise 1 — Resume vs Start Fresh Decision

In [ ]:
# ============ TODO ============
# Build a function `should_resume_or_fresh(session, changed_files, hours_since_last)`
# that returns a recommendation.
#
# Rules:
#   1. If no files changed AND hours_since_last < 4 → "resume"
#   2. If 1-2 files changed AND hours_since_last < 8 → "resume_with_notification"
#      (resume, but prepend a change notification message)
#   3. If 3+ files changed OR hours_since_last >= 24 → "start_fresh_with_summary"
#      (start new session, inject summary of old session)
#   4. If hours_since_last >= 8 AND any files changed → "start_fresh_with_summary"
#
# Return a dict with:
#   - "action": one of the three strings above
#   - "reason": human-readable explanation
#   - "summary": session.get_summary() if action is "start_fresh_with_summary", else None

def should_resume_or_fresh(session, changed_files, hours_since_last):
    # YOUR CODE HERE
    pass

# Test cases:
# s = manager.sessions[list(manager.sessions.keys())[0]]  # grab any session
# print(should_resume_or_fresh(s, [], 1))  # → resume
# print(should_resume_or_fresh(s, ["file_a.py"], 3))  # → resume_with_notification
# print(should_resume_or_fresh(s, ["a.py", "b.py", "c.py"], 2))  # → start_fresh
# print(should_resume_or_fresh(s, ["a.py"], 10))  # → start_fresh
# print(should_resume_or_fresh(s, [], 30))  # → start_fresh
# ==============================

### Exercise 1 — Solution

In [ ]:
def should_resume_or_fresh(session, changed_files, hours_since_last):
    """Decide whether to resume, resume with notification, or start fresh."""
    num_changed = len(changed_files)

    # Rule 4: Long time + any changes → start fresh
    if hours_since_last >= 8 and num_changed > 0:
        return {
            "action": "start_fresh_with_summary",
            "reason": f"{hours_since_last}h since last session AND {num_changed} files changed — context too stale",
            "summary": session.get_summary(),
        }

    # Rule 3: Many files changed OR very long time
    if num_changed >= 3 or hours_since_last >= 24:
        reason = (f"{num_changed} files changed" if num_changed >= 3
                  else f"{hours_since_last}h since last session")
        return {
            "action": "start_fresh_with_summary",
            "reason": f"{reason} — context too stale to resume safely",
            "summary": session.get_summary(),
        }

    # Rule 2: Few files changed, recent session
    if 1 <= num_changed <= 2 and hours_since_last < 8:
        return {
            "action": "resume_with_notification",
            "reason": f"{num_changed} file(s) changed, but session is recent ({hours_since_last}h) — "
                      f"resume and notify about changes",
            "summary": None,
        }

    # Rule 1: No changes, recent session
    if num_changed == 0 and hours_since_last < 4:
        return {
            "action": "resume",
            "reason": f"No files changed, session is fresh ({hours_since_last}h) — safe to resume",
            "summary": None,
        }

    # Edge cases: no changes but somewhat stale
    if num_changed == 0 and hours_since_last < 24:
        return {
            "action": "resume",
            "reason": f"No files changed — safe to resume even though {hours_since_last}h has passed",
            "summary": None,
        }

    return {
        "action": "start_fresh_with_summary",
        "reason": "Default: context staleness uncertain",
        "summary": session.get_summary(),
    }

# Test
s = session  # from earlier
test_cases = [
    ([], 1, "No changes, 1h"),
    (["file_a.py"], 3, "1 file changed, 3h"),
    (["a.py", "b.py", "c.py"], 2, "3 files changed, 2h"),
    (["a.py"], 10, "1 file changed, 10h"),
    ([], 30, "No changes, 30h"),
    (["a.py", "b.py"], 6, "2 files changed, 6h"),
]

print(f"{'Scenario':<30} {'Action':<30} {'Reason'}")
print("=" * 100)
for changed, hours, desc in test_cases:
    result = should_resume_or_fresh(s, changed, hours)
    print(f"  {desc:<28} {result['action']:<30} {result['reason'][:50]}")

In [ ]:
#@title 🎧 Listen: Exercise2
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_09_exercise2.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## Exercise 2 — Implement Crash Recovery

In [ ]:
# ============ TODO ============
# Simulate a crash scenario and implement recovery.
#
# 1. Create a new session called "data-migration"
# 2. Add these messages simulating work in progress:
#    - user: "Migrate customer data from old_db to new_db"
#    - assistant: "Reading old_db schema..." (with a tool call to read_schema)
#    - [tool_result for read_schema]
#    - assistant: "Schema mapped. Now migrating customers table..."
#      (with a tool call to migrate_table — this is the INCOMPLETE call)
#    - NO tool_result for migrate_table — simulating the crash
#
# 3. Set session status to "crashed"
# 4. Use manager.recover_crashed_session("data-migration")
# 5. Verify that the recovery detects the incomplete "migrate_table" call
#
# Hint: For the assistant message with a tool call, use a list as content:
#   [{"type": "text", "text": "..."}, {"type": "tool_use", "name": "migrate_table", ...}]

# YOUR CODE HERE
# ==============================

### Exercise 2 — Solution

In [ ]:
# Create session
crash_session = manager.create_session("data-migration")

# Simulate work
crash_session.add_message("user", "Migrate customer data from old_db to new_db")

# Assistant reads schema (complete tool call)
crash_session.add_message("assistant", [
    {"type": "text", "text": "Reading old_db schema..."},
    {"type": "tool_use", "name": "read_schema", "id": "tool_001", "input": {"db": "old_db"}}
])
crash_session.add_message("user", [
    {"type": "tool_result", "tool_use_id": "tool_001",
     "content": '{"tables": ["customers", "orders", "products"]}'}
])
crash_session.add_tool_result("read_schema", {"db": "old_db"},
                              '{"tables": ["customers", "orders", "products"]}')

# Assistant starts migration (INCOMPLETE — crash happens here)
crash_session.add_message("assistant", [
    {"type": "text", "text": "Schema mapped. Now migrating customers table..."},
    {"type": "tool_use", "name": "migrate_table", "id": "tool_002",
     "input": {"table": "customers", "source": "old_db", "target": "new_db"}}
])
# NO tool_result — crash happened before completion

# Simulate crash
crash_session.status = "crashed"
print(f"Session crashed!")
print(f"  Messages: {len(crash_session.messages)}")
print(f"  Last message role: {crash_session.messages[-1].role}")

# Recover
print(f"\n--- Recovering ---\n")
recovered = manager.recover_crashed_session("data-migration")

if recovered:
    incomplete = recovered.metadata.get("incomplete_tools", [])
    print(f"\nRecovery analysis:")
    print(f"  Incomplete tools: {incomplete}")
    if incomplete:
        print(f"  Action: Re-execute {incomplete} before continuing")
        print(f"  The agent should call migrate_table again with the same parameters")

In [ ]:
#@title 🎧 Listen: Exercise3
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_10_exercise3.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## Exercise 3 — Build a Session Summary Injector

In [ ]:
# ============ TODO ============
# Build a function `start_fresh_with_summary(manager, old_session_name, new_task_message)`
# that:
#   1. Gets the old session and generates its summary
#   2. Creates a new session called "{old_name}-fresh"
#   3. Injects the summary as a system-level context message
#   4. Adds the new task message as the first user message
#   5. Returns the new session
#
# The injected summary should be formatted as:
#   "Context from previous session:\n{summary}\n\nIMPORTANT: The above is a summary
#    of a previous session. File contents may have changed. Do NOT rely on cached
#    tool results from the previous session — re-read any files you need."

def start_fresh_with_summary(manager, old_session_name, new_task_message):
    # YOUR CODE HERE
    pass

# Test:
# new_session = start_fresh_with_summary(
#     manager, "debug-auth-module",
#     "Continue fixing the auth bug. Try a different approach."
# )
# print(f"New session: {new_session.name}")
# print(f"Messages: {len(new_session.messages)}")
# for msg in new_session.messages:
#     print(f"  [{msg.role}]: {str(msg.content)[:80]}...")
# ==============================

### Exercise 3 — Solution

In [ ]:
def start_fresh_with_summary(manager, old_session_name, new_task_message):
    """Start a fresh session with an injected summary from an old session."""
    # Get old session
    old_id = manager.name_index.get(old_session_name)
    if not old_id:
        print(f"Old session '{old_session_name}' not found.")
        return None

    old_session = manager.sessions[old_id]
    summary = old_session.get_summary()

    # Create new session
    new_name = f"{old_session_name}-fresh"
    new_session = manager.create_session(new_name, verbose=False)

    # Inject summary as context
    context_message = (
        f"Context from previous session:\n{summary}\n\n"
        f"IMPORTANT: The above is a summary of a previous session. "
        f"File contents may have changed. Do NOT rely on cached tool results "
        f"from the previous session — re-read any files you need."
    )
    new_session.add_message("system", context_message)

    # Add the new task
    new_session.add_message("user", new_task_message)

    # Record lineage
    new_session.metadata["previous_session"] = old_session_name
    new_session.metadata["started_fresh"] = True

    print(f"Created fresh session '{new_name}' with injected summary")
    print(f"  Previous session: {old_session_name}")
    print(f"  Summary length: {len(summary)} chars")
    return new_session

# Test
new_session = start_fresh_with_summary(
    manager, "debug-auth-module",
    "Continue fixing the auth bug. Try a different approach — "
    "consider adding a token validation middleware."
)

if new_session:
    print(f"\nNew session messages:")
    for i, msg in enumerate(new_session.messages):
        content_preview = str(msg.content)[:100]
        print(f"  [{i}] {msg.role}: {content_preview}...")

In [ ]:
#@title 🎧 Listen: Exercise4
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_11_exercise4.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## Exercise 4 — Fork Comparison Framework

In [ ]:
# ============ TODO ============
# Build a function `compare_forks(manager, fork_names)` that takes a list
# of forked session names and produces a comparison report.
#
# For each fork, report:
#   - Number of messages after the fork point
#   - Number of tool calls after the fork point
#   - Number of files modified (tool results with "edit_file" or "create_file")
#   - Test results (look for tool_results with tool_name "run_tests")
#   - The last assistant message (as a summary of what was accomplished)
#
# Return a list of dicts, one per fork, sorted by "test_pass_rate" descending.

def compare_forks(manager, fork_names):
    # YOUR CODE HERE
    pass

# Test with the forks we created earlier:
# results = compare_forks(manager, ["fix-via-refactor", "fix-via-validation"])
# for r in results:
#     print(f"  {r['name']}: {r['test_pass_rate']}% pass rate, {r['files_modified']} files modified")
# ==============================

### Exercise 4 — Solution

In [ ]:
def compare_forks(manager, fork_names):
    """Compare forked sessions and produce a ranked report."""
    comparisons = []

    for name in fork_names:
        sid = manager.name_index.get(name)
        if not sid:
            print(f"  Warning: session '{name}' not found")
            continue

        session = manager.sessions[sid]
        fork_point = session.fork_point_index or 0

        # Messages after fork
        post_fork_messages = session.messages[fork_point:]
        post_fork_tools = [tr for i, tr in enumerate(session.tool_results)
                           if i >= 0]  # simplified — all tools in fork are post-fork

        # Files modified
        files_modified = len([tr for tr in session.tool_results
                              if tr.tool_name in ("edit_file", "create_file")])

        # Test results
        test_results = None
        test_pass_rate = 0
        for tr in session.tool_results:
            if tr.tool_name == "run_tests":
                test_results = json.loads(tr.result)
                total = test_results.get("passed", 0) + test_results.get("failed", 0)
                test_pass_rate = (test_results["passed"] / total * 100) if total > 0 else 0

        # Last assistant message
        last_assistant = ""
        for msg in reversed(session.messages):
            if msg.role == "assistant" and isinstance(msg.content, str):
                last_assistant = msg.content
                break

        comparisons.append({
            "name": name,
            "messages_after_fork": len(post_fork_messages),
            "tool_calls_after_fork": len(post_fork_tools),
            "files_modified": files_modified,
            "test_pass_rate": test_pass_rate,
            "test_details": test_results,
            "last_finding": last_assistant[:100],
        })

    # Sort by test pass rate (descending)
    comparisons.sort(key=lambda x: x["test_pass_rate"], reverse=True)
    return comparisons

# Compare our forks
results = compare_forks(manager, ["fix-via-refactor", "fix-via-validation"])

print(f"\n{'='*60}")
print(f"FORK COMPARISON REPORT")
print(f"{'='*60}")
print(f"\n{'Branch':<25} {'Pass Rate':<12} {'Files':<8} {'Tool Calls':<12} {'Winner?'}")
print("-" * 70)
for i, r in enumerate(results):
    winner = " *** BEST" if i == 0 else ""
    print(f"  {r['name']:<23} {r['test_pass_rate']:>5.1f}%     "
          f"{r['files_modified']:<8} {r['tool_calls_after_fork']:<12}{winner}")

if results:
    print(f"\nRecommendation: Use '{results[0]['name']}' ({results[0]['test_pass_rate']:.0f}% pass rate)")
    print(f"  Summary: {results[0]['last_finding']}...")

In [ ]:
#@title 🎧 Listen: Exercise5
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_12_exercise5.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## Exercise 5 — Session Lifecycle State Machine

In [ ]:
# ============ TODO ============
# Build a function `validate_transition(current_status, action)` that enforces
# valid session state transitions.
#
# Valid transitions:
#   "active"    → "completed" (via "complete")
#   "active"    → "crashed"   (via "crash")
#   "active"    → "forked"    (via "fork") — marks the SOURCE as forked
#   "completed" → "active"    (via "resume")
#   "crashed"   → "active"    (via "recover")
#   "forked"    → "active"    (via "resume") — can still resume the source
#
# Invalid transitions should return {"valid": False, "reason": "..."}
# Valid transitions should return {"valid": True, "new_status": "..."}

VALID_TRANSITIONS = {
    # YOUR CODE HERE — define the state machine as a dict
}

def validate_transition(current_status, action):
    # YOUR CODE HERE
    pass

# Test all transitions:
# test_transitions = [
#     ("active", "complete"),   # valid
#     ("active", "crash"),      # valid
#     ("active", "fork"),       # valid
#     ("active", "resume"),     # invalid
#     ("completed", "resume"),  # valid
#     ("crashed", "recover"),   # valid
#     ("crashed", "complete"),  # invalid
#     ("forked", "resume"),     # valid
# ]
# for status, action in test_transitions:
#     result = validate_transition(status, action)
#     v = "VALID" if result["valid"] else "INVALID"
#     print(f"  {status} --{action}--> {result.get('new_status', '?')}: {v}")
# ==============================

### Exercise 5 — Solution

In [ ]:
VALID_TRANSITIONS = {
    ("active", "complete"): "completed",
    ("active", "crash"): "crashed",
    ("active", "fork"): "forked",
    ("completed", "resume"): "active",
    ("crashed", "recover"): "active",
    ("forked", "resume"): "active",
}

def validate_transition(current_status, action):
    """Validate a session state transition."""
    key = (current_status, action)
    if key in VALID_TRANSITIONS:
        return {"valid": True, "new_status": VALID_TRANSITIONS[key]}
    else:
        # Find valid actions for current status
        valid_actions = [a for (s, a) in VALID_TRANSITIONS if s == current_status]
        return {
            "valid": False,
            "reason": f"Cannot '{action}' a session in '{current_status}' state. "
                      f"Valid actions: {valid_actions}"
        }

# Test
test_transitions = [
    ("active", "complete"),
    ("active", "crash"),
    ("active", "fork"),
    ("active", "resume"),
    ("completed", "resume"),
    ("completed", "crash"),
    ("crashed", "recover"),
    ("crashed", "complete"),
    ("forked", "resume"),
    ("forked", "fork"),
]

print(f"{'From':<12} {'Action':<10} {'To':<12} {'Valid?':<8} {'Notes'}")
print("=" * 65)
for status, action in test_transitions:
    result = validate_transition(status, action)
    if result["valid"]:
        print(f"  {status:<10} {action:<10} {result['new_status']:<12} OK")
    else:
        print(f"  {status:<10} {action:<10} {'?':<12} BLOCKED  {result['reason'][:35]}...")

In [ ]:
#@title 🎧 Listen: Visualization
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_13_visualization.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 7. Visualization — Session Lifecycle and Fork Tree

In [ ]:
def visualize_session_lifecycle(manager):
    """Visualize the session state machine and current session tree."""
    print("=" * 65)
    print("          SESSION LIFECYCLE STATE MACHINE")
    print("=" * 65)
    print()
    print("  ┌──────────┐  complete   ┌───────────┐")
    print("  │  ACTIVE  │────────────>│ COMPLETED  │")
    print("  │          │             │            │")
    print("  │          │<────────────│            │")
    print("  └────┬─────┘   resume    └────────────┘")
    print("       │")
    print("       │ crash")
    print("       ▼")
    print("  ┌──────────┐")
    print("  │ CRASHED  │")
    print("  │          │── recover ──> back to ACTIVE")
    print("  └──────────┘")
    print()
    print("  ┌──────────┐")
    print("  │  ACTIVE  │── fork ──> FORKED (source)")
    print("  │          │             + new ACTIVE (fork copy)")
    print("  └──────────┘")
    print()

    # Show current session tree
    print("  CURRENT SESSION TREE")
    print("  " + "-" * 40)
    print()

    # Find root sessions (no parent)
    roots = [s for s in manager.sessions.values() if s.parent_session_id is None]

    for root in roots:
        _print_session_tree(manager, root, indent=2)

def _print_session_tree(manager, session, indent=0):
    """Recursively print a session and its forks."""
    prefix = " " * indent
    status_icon = {
        "active": "[*]",
        "completed": "[v]",
        "crashed": "[!]",
        "forked": "[>]",
    }.get(session.status, "[?]")

    msgs = len(session.messages)
    tools = len(session.tool_results)
    print(f"{prefix}{status_icon} {session.name} ({msgs} msgs, {tools} tools)")

    # Find children (forked sessions)
    children = [s for s in manager.sessions.values()
                if s.parent_session_id == session.session_id]
    for child in children:
        print(f"{prefix}  └── ", end="")
        _print_session_tree(manager, child, indent + 6)

visualize_session_lifecycle(manager)

In [ ]:
#@title 🎧 Listen: Mini Project
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_14_mini_project.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 8. Mini-Project — Complete Agent Session Infrastructure

Build a complete session-aware agent that handles create, resume, fork, crash recovery, and stale context detection.

In [ ]:
class SessionAwareAgent:
    """A complete agent with session management capabilities."""

    def __init__(self):
        self.manager = SessionManager()
        self.active_session: Optional[Session] = None

    def start(self, task: str, session_name: str = "", verbose: bool = True) -> str:
        """Start a new task with a fresh session."""
        name = session_name or f"task-{uuid.uuid4().hex[:4]}"
        self.active_session = self.manager.create_session(name, verbose=verbose)
        self.active_session.add_message("user", task)

        if verbose:
            print(f"  Task: {task}")
        return self.active_session.name

    def resume(self, session_name: str, changed_files: List[str] = None,
               verbose: bool = True) -> Optional[str]:
        """Resume a session with stale context checking."""
        changed_files = changed_files or []

        session = self.manager.resume_session(session_name, verbose=verbose)
        if not session:
            return None

        # Check for stale context
        if changed_files:
            stale_check = self.manager.detect_stale_context(session, changed_files)
            if verbose:
                print(f"\n  Stale context check: {stale_check['recommendation']}")

            if stale_check["is_stale"]:
                # Notify the session about changes
                notification = (
                    "FILES CHANGED since last session:\n" +
                    "\n".join(f"  - {f}" for f in stale_check["stale_files"]) +
                    "\nRe-read these files before continuing."
                )
                session.add_message("user", notification)
                if verbose:
                    print(f"  Injected change notification for {len(stale_check['stale_files'])} files")

        self.active_session = session
        return session.name

    def fork(self, approach_description: str, fork_name: str = "",
             verbose: bool = True) -> Optional[str]:
        """Fork current session for parallel exploration."""
        if not self.active_session:
            print("No active session to fork.")
            return None

        forked = self.manager.fork_session(
            self.active_session.name, fork_name, verbose=verbose
        )
        if forked:
            forked.add_message("user", f"New approach: {approach_description}")
            return forked.name
        return None

    def simulate_work(self, steps: List[Dict], verbose: bool = True):
        """Simulate agent work (reading files, calling tools, etc.)."""
        if not self.active_session:
            print("No active session.")
            return

        for step in steps:
            action = step.get("action", "message")

            if action == "read_file":
                path = step["path"]
                self.active_session.files_read.append(path)
                self.active_session.add_tool_result(
                    "read_file", {"path": path},
                    json.dumps({"content": step.get("content", "file content...")}),
                    source_file=path
                )
                if verbose:
                    print(f"  Read: {path}")

            elif action == "assistant_message":
                self.active_session.add_message("assistant", step["text"])
                if verbose:
                    print(f"  Agent: {step['text'][:60]}...")

            elif action == "tool_call":
                self.active_session.add_tool_result(
                    step["tool"], step.get("input", {}),
                    json.dumps(step.get("result", {"success": True})),
                    source_file=step.get("source_file")
                )
                if verbose:
                    print(f"  Tool: {step['tool']}({json.dumps(step.get('input', {}))})")

    def simulate_crash(self, verbose: bool = True):
        """Simulate a crash."""
        if self.active_session:
            self.active_session.status = "crashed"
            if verbose:
                print(f"\n  *** SESSION CRASHED: {self.active_session.name} ***")
            self.active_session = None

    def recover(self, session_name: str, verbose: bool = True) -> Optional[str]:
        """Recover a crashed session."""
        session = self.manager.recover_crashed_session(session_name, verbose=verbose)
        if session:
            self.active_session = session
            return session.name
        return None

    def status(self):
        """Print current status."""
        print(f"\n{'='*50}")
        print(f"AGENT STATUS")
        print(f"{'='*50}")
        active = self.active_session
        if active:
            print(f"  Active session: {active.name}")
            print(f"  Messages: {len(active.messages)}")
            print(f"  Tool calls: {len(active.tool_results)}")
            print(f"  Files read: {active.files_read}")
        else:
            print(f"  No active session")
        print(f"  Total sessions: {len(self.manager.sessions)}")
        self.manager.list_sessions()


# Demo: Full lifecycle
agent = SessionAwareAgent()

# 1. Start a task
print("="*60)
print("PHASE 1: Start a new debugging task")
print("="*60)
agent.start("Fix the intermittent auth failures in production", "debug-prod-auth")
agent.simulate_work([
    {"action": "read_file", "path": "src/auth/login.py", "content": "def login(): ..."},
    {"action": "assistant_message", "text": "Found retry logic without backoff in login.py. Investigating token module next."},
    {"action": "read_file", "path": "src/auth/token.py", "content": "def refresh(): cached..."},
    {"action": "assistant_message", "text": "Root cause: stale tokens served from cache. Two possible fixes identified."},
])

# 2. Fork for parallel exploration
print(f"\n{'='*60}")
print("PHASE 2: Fork for two approaches")
print("="*60)
fork_a = agent.fork("Refactor token refresh to check expiry before cache", "approach-refactor")
fork_b = agent.fork("Add validation middleware before auth module", "approach-middleware")
print(f"  Created branches: {fork_a}, {fork_b}")

# 3. Simulate crash
print(f"\n{'='*60}")
print("PHASE 3: Simulate crash during work")
print("="*60)
agent.simulate_crash()

# 4. Recover
print(f"\n{'='*60}")
print("PHASE 4: Recover crashed session")
print("="*60)
agent.recover("debug-prod-auth")

# 5. Resume with stale context
print(f"\n{'='*60}")
print("PHASE 5: Resume with file changes detected")
print("="*60)
agent.active_session.status = "completed"  # close current
agent.resume("debug-prod-auth", changed_files=["src/auth/login.py", "src/auth/config.py"])

# Final status
agent.status()

# Show session tree
print()
visualize_session_lifecycle(agent.manager)

In [ ]:
#@title 🎧 Listen: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/05_15_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 9. Key Takeaways

**Exam-critical concepts for Task Statement 1.7 (Session Management):**

- **`--resume` with session names** restores full conversation history — use it when the interruption was brief and files have not changed significantly
- **`fork_session`** creates a branch with copied context for parallel exploration — each fork evolves independently, the original session is never modified
- **Stale context is dangerous** — if files have changed since the last session, tool results from the previous session may reference code that no longer exists. Always inform the agent about changes
- **Resume vs start fresh:** resume when context is current (few changes, recent session). Start fresh with an injected summary when context is stale (many files changed, long time elapsed, or tool results would be misleading)
- **Crash recovery** must detect incomplete tool calls — tools that were called but never returned results. These need to be re-executed before the agent can continue
- **Injected summaries** provide the best of both worlds — the agent gets the key findings from the old session without the stale tool results

In [ ]:
print("Notebook 5 complete!")
print()
print("Key exam decision framework:")
print("  Brief interruption, no file changes → --resume")
print("  Parallel exploration needed → fork_session")
print("  Many files changed or long gap → start fresh + injected summary")
print("  Crash detected → recover + re-execute incomplete tool calls")
print()
print("Congratulations! You have completed all 5 Agentic Architecture labs.")